# 📊 Visualization — B1 Descriptive và B2 Distribution

Notebook đọc dữ liệu feature từ `data/processed/commits_features.csv`. Tất cả hình được lưu vào `reports/figures/` để chèn vào báo cáo. Ngày giờ được phân tích theo UTC.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # chạy được cả khi không có GUI/Tkinter
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
DATA_PATH = Path('../data/processed/commits_features.csv')
if not DATA_PATH.exists(): DATA_PATH = Path('../data/processed/cleaned_commits_data_final.csv')
FIG_DIR = Path('../reports/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH, low_memory=False)
df['commit_date'] = pd.to_datetime(df['commit_date'], utc=True, errors='coerce')
if 'change_commit_date' in df: df['change_commit_date'] = pd.to_datetime(df['change_commit_date'], utc=True, errors='coerce')
print(f'✅ Đã tải {DATA_PATH}: {len(df):,} dòng × {df.shape[1]} cột')

def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout(); plt.savefig(path, dpi=160, bbox_inches='tight'); print(f'Đã lưu {path}')
    plt.show(); plt.close()

## B1 — Descriptive Visualization

In [ ]:
# B1.1 — Số lượng theo năm
year_counts = df.groupby('year').size().sort_index()
plt.figure(figsize=(8, 4.5)); ax = sns.barplot(x=year_counts.index.astype(str), y=year_counts.values, color='#16a34a')
ax.set(title='Số lượng AI commit theo năm', xlabel='Năm', ylabel='Số dòng')
for container in ax.containers: ax.bar_label(container, fmt='%.0f')
savefig('B1_01_commits_by_year.png')

# B1.2 — Tỷ lệ ngôn ngữ
lang = df['language'].value_counts()
plt.figure(figsize=(5.5, 5.5)); plt.pie(lang.values, labels=[f'{k}: {v:,}' for k,v in lang.items()], autopct='%.1f%%', startangle=90, colors=['#3776AB', '#F7DF1E'])
plt.title('Tỷ lệ ngôn ngữ lập trình'); savefig('B1_02_language_share.png')

# B1.3 — Heatmap giờ × thứ
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heat = pd.crosstab(df['day_of_week'], df['hour']).reindex(index=day_order, columns=range(24), fill_value=0)
plt.figure(figsize=(16, 4.5)); sns.heatmap(heat, cmap='turbo', linewidths=.3); plt.title('Mật độ AI commit theo giờ và thứ (UTC)'); plt.xlabel('Giờ'); plt.ylabel('Ngày trong tuần'); savefig('B1_03_hour_day_heatmap.png')

# B1.4 — Tăng trưởng theo tháng
monthly = df.groupby('year_month').size().sort_index()
plt.figure(figsize=(13, 4.5)); plt.plot(monthly.index, monthly.values, marker='o', linewidth=1.8, color='#2563eb'); plt.xticks(rotation=60); plt.title('Tăng trưởng AI commit theo tháng'); plt.xlabel('Tháng'); plt.ylabel('Số dòng'); savefig('B1_04_monthly_growth.png')

# B1.5 — Tỷ lệ is_modified (donut)
modified = df['is_modified'].value_counts().reindex([False, True], fill_value=0)
plt.figure(figsize=(5.5, 5.5)); plt.pie(modified.values, labels=['Chưa sửa','Đã sửa'], autopct='%.1f%%', startangle=90, colors=['#22c55e','#ef4444'], wedgeprops={'width':.42, 'edgecolor':'white'}); plt.title('Code AI có bị sửa không?'); savefig('B1_05_modified_donut.png')

# B1.6 — Top 15 repo theo số interaction
top_repo = df['repo'].value_counts().head(15).sort_values()
plt.figure(figsize=(9, 6)); sns.barplot(x=top_repo.values, y=top_repo.index, color='#7c3aed'); plt.title('Top 15 repository có nhiều AI interaction nhất'); plt.xlabel('Số dòng'); plt.ylabel('Repository'); savefig('B1_06_top_repositories.png')

# B1.7 — Công cụ AI
tool = df['ai_tool'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(9, 4.8)); sns.barplot(x=tool.values, y=tool.index, color='#0891b2'); plt.title('Phân bố công cụ AI (heuristic)'); plt.xlabel('Số dòng'); plt.ylabel('AI tool'); savefig('B1_07_ai_tool.png')

# B1.8 — Mức đóng góp / ai_verb
verb = df['ai_verb'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(8, 4.5)); sns.barplot(x=verb.values, y=verb.index, color='#f97316'); plt.title('Mức đóng góp AI (heuristic)'); plt.xlabel('Số dòng'); plt.ylabel('ai_verb'); savefig('B1_08_ai_verb.png')

# B1.9 — Python vs JS theo tháng (stacked area)
lang_month = pd.crosstab(df['year_month'], df['language']).sort_index()
plt.figure(figsize=(13, 4.8)); plt.stackplot(lang_month.index, [lang_month.get(c, pd.Series(0, index=lang_month.index)) for c in ['python','js']], labels=['python','js'], colors=['#3776AB','#F7DF1E']); plt.xticks(rotation=60); plt.title('Python vs JavaScript theo tháng'); plt.xlabel('Tháng'); plt.ylabel('Số dòng'); plt.legend(loc='upper left'); savefig('B1_09_language_month_area.png')

# B1.10 — Task type
task = df['task_type'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(9, 4.8)); sns.barplot(x=task.values, y=task.index, color='#db2777'); plt.title('AI được dùng để làm loại công việc nào?'); plt.xlabel('Số dòng'); plt.ylabel('task_type'); savefig('B1_10_task_type.png')

# B1.11 — Task type × is_modified (tỷ lệ trong từng task)
task_mod = pd.crosstab(df['task_type'], df['is_modified'], normalize='index').reindex(columns=[False, True], fill_value=0).sort_values(True)
ax = task_mod.plot(kind='barh', stacked=True, figsize=(9, 5), color=['#22c55e','#ef4444']); ax.set(title='Tỷ lệ code bị sửa theo loại công việc', xlabel='Tỷ lệ', ylabel='task_type'); ax.legend(['Chưa sửa','Đã sửa'], loc='lower right'); savefig('B1_11_task_modified_stacked.png')

# B1.12 — Reliability theo ai_tool: tỷ lệ chưa bị sửa
reliability = df.groupby('ai_tool').agg(n=('is_modified','size'), modified_rate=('is_modified','mean')).assign(reliability=lambda x: 1-x['modified_rate']).sort_values('reliability')
ax = reliability['reliability'].mul(100).plot(kind='barh', figsize=(9, 5), color='#16a34a'); ax.set(title='Reliability theo AI tool (100% − tỷ lệ bị sửa)', xlabel='Reliability (%)', ylabel='ai_tool'); savefig('B1_12_tool_reliability.png')
display(reliability.sort_values('reliability', ascending=False).assign(reliability_pct=lambda x: (x.reliability*100).round(2)))

## B2 — Distribution Analysis

In [ ]:
# Chỉ dùng số ngày hợp lệ (đã loại ngày âm ở notebook 03)
days = df.loc[df['days_to_modification'].notna() & (df['days_to_modification'] >= 0), 'days_to_modification']
# B2.1 — Histogram + KDE
plt.figure(figsize=(9, 4.8)); sns.histplot(days, bins=50, kde=True, color='#2563eb'); plt.title('Phân phối days_to_modification'); plt.xlabel('Số ngày đến lần sửa đầu tiên'); plt.ylabel('Số dòng'); savefig('B2_01_days_hist_kde.png')

# B2.2 — Code lines theo ngôn ngữ (boxplot + violin)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8)); sns.boxplot(data=df, x='language', y='code_lines', ax=axes[0], showfliers=False); axes[0].set_title('Boxplot code_lines'); sns.violinplot(data=df, x='language', y='code_lines', ax=axes[1], cut=0); axes[1].set_title('Violin code_lines'); fig.suptitle('Phân phối kích thước code theo ngôn ngữ'); savefig('B2_02_code_lines_language.png')

# B2.3 — Q-Q plot trên log1p(days), phù hợp với phân phối lệch phải
plt.figure(figsize=(6, 6)); stats.probplot(np.log1p(days), dist='norm', plot=plt); plt.title('Q-Q plot log1p(days_to_modification)'); savefig('B2_03_qq_days.png')

# B2.4 — Correlation matrix
corr_cols = ['year','month','hour','is_weekend','is_modified','days_to_modification','code_lines','code_chars','repo_commit_count']
corr = df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, square=True); plt.title('Correlation matrix của các numeric features'); savefig('B2_04_correlation_matrix.png')
print(f'✅ B2.1 dùng {len(days):,} dòng days_to_modification hợp lệ')